<a href="https://colab.research.google.com/github/Park-gpow/bigdataAn/blob/main/An01(04_20).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
#    Google 런타임용 설정 + raw 로드
#    실행 위치: Colab
#    결과 변수: train_raw, val_raw, test_raw
#    무족건 실행 후 진행
# =========================================================

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/빅데이터 사례연구")

TN_DIR = BASE_DIR / "Tn"
VAL_DIR = BASE_DIR / "Val"
TS_DIR = BASE_DIR / "Ts"

TRAIN_RAW_PATH = TN_DIR / "train_dev_raw.csv"
VAL_RAW_PATH = VAL_DIR / "val_dev_raw.csv"
TEST_RAW_PATH = TS_DIR / "test_raw.csv"

train_raw = pd.read_csv(TRAIN_RAW_PATH)
val_raw = pd.read_csv(VAL_RAW_PATH)
test_raw = pd.read_csv(TEST_RAW_PATH)

print("[확인 완료] 구글런타임 준비완료")
print("shapes:", train_raw.shape, val_raw.shape, test_raw.shape)

Mounted at /content/drive
[확인 완료] 구글런타임 준비완료
shapes: (105432, 48) (26358, 48) (16474, 48)


In [ ]:
# =========================================================
#    로컬 전달용 다운로드 셀
#    실행 위치: Colab
#    역할: Tn / Val / Ts raw 파일 3개를 zip으로 다운로드
#    새로운 환경이나 등등 진행여부가 필요시 진행.
# =========================================================

from google.colab import files
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/Tn_Val_Ts_raw.zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(TRAIN_RAW_PATH, arcname="Tn/train_dev_raw.csv")
    zf.write(VAL_RAW_PATH, arcname="Val/val_dev_raw.csv")
    zf.write(TEST_RAW_PATH, arcname="Ts/test_raw.csv")

print("[확인 완료] 로컬 준비완료")
print("다운로드 파일:", ZIP_PATH)
print(r"압축 해제 위치: C:\Users\data\Desktop\4학년 잠시 저장 파일\임시 데이터셋 다운")

files.download(str(ZIP_PATH))

In [ ]:
# =========================================================
# 3. 로컬 환경용 설정 + raw 로드
#    실행 위치: 로컬 Python / Jupyter / VS Code
#    결과 변수: train_raw, val_raw, test_raw
# =========================================================

from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\data\Desktop\4학년 잠시 저장 파일\임시 데이터셋 다운")

TN_DIR = BASE_DIR / "Tn"
VAL_DIR = BASE_DIR / "Val"
TS_DIR = BASE_DIR / "Ts"

TRAIN_RAW_PATH = TN_DIR / "train_dev_raw.csv"
VAL_RAW_PATH = VAL_DIR / "val_dev_raw.csv"
TEST_RAW_PATH = TS_DIR / "test_raw.csv"

train_raw = pd.read_csv(TRAIN_RAW_PATH)
val_raw = pd.read_csv(VAL_RAW_PATH)
test_raw = pd.read_csv(TEST_RAW_PATH)

print("[확인 완료] 로컬 준비완료")
print("shapes:", train_raw.shape, val_raw.shape, test_raw.shape)

In [2]:
# =========================================================
# 1. 컬럼명 정리
# =========================================================
import os
import json
import numpy as np
import pandas as pd

RANDOM_STATE = 42

rename_map = {
    "discharge_capacity (mAh/g)": "discharge_capacity"
}

train_df = train_raw.rename(columns=rename_map).copy()
val_df   = val_raw.rename(columns=rename_map).copy()
test_df  = test_raw.rename(columns=rename_map).copy()

print("[컬럼명 정리 후 shapes]")
print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

# =========================================================
# 2. 기본 점검
#    - material_id 중복
#    - 결측치 존재 여부
# =========================================================
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n[{name} material_id 중복 개수]")
    if "material_id" in df.columns:
        print(df["material_id"].duplicated().sum())
    else:
        print("material_id 컬럼 없음")

    print(f"\n[{name} 결측치 총개수]")
    print(int(df.isna().sum().sum()))

    print(f"\n[{name} 결측치 상위 컬럼]")
    print(df.isna().sum().sort_values(ascending=False).head(10))

# 본문과 맞추기 위해 결측치가 없다는 전제 확인
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    total_missing = int(df.isna().sum().sum())
    if total_missing > 0:
        raise ValueError(
            f"[{name}] 결측치가 {total_missing}개 확인되었습니다. "
            "현재 논문 본문에는 결측치가 없다고 되어 있으므로, "
            "본문을 수정하거나 결측치 처리 로직을 추가해야 합니다."
        )

print("\n[확인 완료] train / val / test 모두 결측치 없음")

# =========================================================
# 3. 반응변수 및 제외 변수 정의
# =========================================================
TARGET = "discharge_capacity"

explicit_exclude_cols = [
    "material_id",
    "remain_capacity",
    "state_of_charge",
    "Strain",
    "Li_fraction",
    "Ni_fraction",
    "Mn_fraction",
    "Co_fraction",
    "dopant_fraction",
    "Co_source",
    "Mn_source",
    "Ni_source",
    "space_group_symbol"
]

existing_explicit_exclude = [c for c in explicit_exclude_cols if c in train_df.columns]

print("\n[명시적 제외 변수]")
print(existing_explicit_exclude)

# =========================================================
# 4. train 기준 상수형 변수 자동 탐지
#    - 명시적 제외와 별개로, 실제로 값이 하나뿐인 변수는 추가 제거
# =========================================================
candidate_predictors = [c for c in train_df.columns if c != TARGET]
constant_cols_auto = [c for c in candidate_predictors if train_df[c].nunique(dropna=False) <= 1]

# 이미 명시적 제외에 들어간 변수는 중복 제거
constant_cols_auto = [c for c in constant_cols_auto if c not in existing_explicit_exclude]

print("\n[train 기준 자동 탐지된 상수형 변수]")
print(constant_cols_auto)

final_exclude_cols = sorted(set(existing_explicit_exclude + constant_cols_auto))

print("\n[최종 제외 변수]")
print(final_exclude_cols)

# =========================================================
# 5. 명백한 입력 오류 점검 및 제거
#    - 음수 불가 변수
#    - 전압범위 역전
# =========================================================
nonnegative_candidates = [
    "sintering_T1(C)",
    "sintering_t1(h)",
    "measurement_T(C)",
    "C-rate",
    "active_proportion",
    "binder_proportion",
    "particle_size(um)",
    "volume",
    "density",
    "interlayer_dist",
    "tm_o_bond_length",
    "discharge_capacity"
]

nonnegative_candidates = [c for c in nonnegative_candidates if c in train_df.columns]

def count_obvious_errors(df):
    report = {}
    for col in nonnegative_candidates:
        report[f"negative_{col}"] = int((df[col] < 0).sum())

    if {"voltage_range(V)_min", "voltage_range(V)_max"}.issubset(df.columns):
        report["reversed_voltage_range"] = int(
            (df["voltage_range(V)_min"] > df["voltage_range(V)_max"]).sum()
        )
    else:
        report["reversed_voltage_range"] = 0

    return report

def remove_obvious_errors(df):
    out = df.copy()

    for col in nonnegative_candidates:
        out = out.loc[out[col] >= 0]

    if {"voltage_range(V)_min", "voltage_range(V)_max"}.issubset(out.columns):
        out = out.loc[out["voltage_range(V)_min"] <= out["voltage_range(V)_max"]]

    return out

print("\n[입력 오류 점검 - train]")
print(json.dumps(count_obvious_errors(train_df), indent=2, ensure_ascii=False))

print("\n[입력 오류 점검 - val]")
print(json.dumps(count_obvious_errors(val_df), indent=2, ensure_ascii=False))

print("\n[입력 오류 점검 - test]")
print(json.dumps(count_obvious_errors(test_df), indent=2, ensure_ascii=False))

train_df = remove_obvious_errors(train_df)
val_df   = remove_obvious_errors(val_df)
test_df  = remove_obvious_errors(test_df)

print("\n[입력 오류 제거 후 shapes]")
print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

# =========================================================
# 6. 희소 범주 통합
#    - train 기준으로 모든 범주형 변수에 적용
#    - 빈도 < 30 또는 비율 < 1% 이면 Other
# =========================================================
def fit_keep_levels(series, min_count=30, min_ratio=0.01):
    s = series.astype(str).fillna("Unknown")
    counts = s.value_counts(dropna=False)
    ratios = counts / len(s)
    keep = counts[(counts >= min_count) & (ratios >= min_ratio)].index.tolist()
    return keep

def apply_keep_levels(series, keep_levels):
    s = series.astype(str).fillna("Unknown")
    return np.where(s.isin(keep_levels), s, "Other")

categorical_cols = train_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in final_exclude_cols and c != TARGET]

print("\n[희소 범주 통합 대상 범주형 변수]")
print(categorical_cols)

keep_levels_map = {}
for col in categorical_cols:
    keep_levels_map[col] = fit_keep_levels(train_df[col], min_count=30, min_ratio=0.01)

for col in categorical_cols:
    train_df[col] = apply_keep_levels(train_df[col], keep_levels_map[col])
    val_df[col]   = apply_keep_levels(val_df[col], keep_levels_map[col])
    test_df[col]  = apply_keep_levels(test_df[col], keep_levels_map[col])

print("\n[희소 범주 통합 완료]")
for col in categorical_cols[:10]:
    print(f"{col}: 유지수준 {len(keep_levels_map[col])}개")

# =========================================================
# 7. 제외 변수 제거
# =========================================================
def drop_excluded(df, exclude_cols):
    keep_cols = [c for c in df.columns if c not in exclude_cols]
    return df[keep_cols].copy()

train_proc = drop_excluded(train_df, final_exclude_cols)
val_proc   = drop_excluded(val_df, final_exclude_cols)
test_proc  = drop_excluded(test_df, final_exclude_cols)

print("\n[제외 변수 제거 후 shapes]")
print("train_proc:", train_proc.shape)
print("val_proc  :", val_proc.shape)
print("test_proc :", test_proc.shape)

# =========================================================
# 8. 로그변환 후보 탐색
#    - 왜도 절댓값 > 1 이고 모든 값이 양수인 수치형 변수
#    - 실제 변환은 아직 적용하지 않고 후보만 저장
# =========================================================
numeric_cols = train_proc.select_dtypes(include=[np.number]).columns.tolist()
numeric_predictors = [c for c in numeric_cols if c != TARGET]

skew_candidates = []
for col in numeric_predictors:
    x = train_proc[col].dropna()
    if len(x) > 0 and (x > 0).all():
        skew_val = x.skew()
        if abs(skew_val) > 1:
            skew_candidates.append({
                "variable": col,
                "skewness": round(float(skew_val), 4)
            })

print("\n[로그변환 후보 변수]")
print(skew_candidates)

# =========================================================
# 9. 이상치 점검표 생성
#    - IQR 기준 플래그만 생성
#    - 기계적 제거는 하지 않음
# =========================================================
outlier_flags = []

for col in numeric_predictors:
    x = train_proc[col].dropna()
    if len(x) == 0:
        continue

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged_n = int(((train_proc[col] < lower) | (train_proc[col] > upper)).sum())

    outlier_flags.append({
        "variable": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "flagged_n": flagged_n
    })

outlier_flag_df = pd.DataFrame(outlier_flags)

# =========================================================
# 10. 모델 입력용 더미변수화
#     - train 기준 컬럼을 val/test에 맞춤
# =========================================================
feature_cols = [c for c in train_proc.columns if c != TARGET]

X_train = pd.get_dummies(train_proc[feature_cols], drop_first=True)
X_val   = pd.get_dummies(val_proc[feature_cols], drop_first=True)
X_test  = pd.get_dummies(test_proc[feature_cols], drop_first=True)

X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_proc[TARGET]
y_val   = val_proc[TARGET]
y_test  = test_proc[TARGET]

print("\n[모델 입력 행렬 크기]")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

# =========================================================
# 11. 저장
# =========================================================
OUT_DIR = BASE_DIR / "processed_lfp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_proc.to_csv(OUT_DIR / "train_processed.csv", index=False, encoding="utf-8-sig")
val_proc.to_csv(OUT_DIR / "val_processed.csv", index=False, encoding="utf-8-sig")
test_proc.to_csv(OUT_DIR / "test_processed.csv", index=False, encoding="utf-8-sig")

X_train.to_csv(OUT_DIR / "X_train_model_ready.csv", index=False, encoding="utf-8-sig")
X_val.to_csv(OUT_DIR / "X_val_model_ready.csv", index=False, encoding="utf-8-sig")
X_test.to_csv(OUT_DIR / "X_test_model_ready.csv", index=False, encoding="utf-8-sig")

y_train.to_csv(OUT_DIR / "y_train.csv", index=False, encoding="utf-8-sig")
y_val.to_csv(OUT_DIR / "y_val.csv", index=False, encoding="utf-8-sig")
y_test.to_csv(OUT_DIR / "y_test.csv", index=False, encoding="utf-8-sig")

outlier_flag_df.to_csv(OUT_DIR / "outlier_flag_summary.csv", index=False, encoding="utf-8-sig")

metadata = {
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "explicit_exclude_cols": existing_explicit_exclude,
    "constant_cols_auto": constant_cols_auto,
    "final_exclude_cols": final_exclude_cols,
    "categorical_cols_collapsed": categorical_cols,
    "rare_category_rule": {
        "min_count": 30,
        "min_ratio": 0.01
    },
    "keep_levels_map": keep_levels_map,
    "skew_candidates": skew_candidates,
    "train_shape_after_processing": train_proc.shape,
    "val_shape_after_processing": val_proc.shape,
    "test_shape_after_processing": test_proc.shape
}

with open(OUT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n[완료] 전처리 및 모델 입력 데이터 저장 완료")
print("저장 위치:", OUT_DIR)

[컬럼명 정리 후 shapes]
train: (105432, 48)
val  : (26358, 48)
test : (16474, 48)

[train material_id 중복 개수]
0

[train 결측치 총개수]
0

[train 결측치 상위 컬럼]
material_id           0
material_structure    0
synthesis_method      0
sintering_T1(C)       0
sintering_t1(h)       0
Li_source             0
Co_source             0
Mn_source             0
Ni_source             0
electrolyte           0
dtype: int64

[val material_id 중복 개수]
0

[val 결측치 총개수]
0

[val 결측치 상위 컬럼]
material_id           0
material_structure    0
synthesis_method      0
sintering_T1(C)       0
sintering_t1(h)       0
Li_source             0
Co_source             0
Mn_source             0
Ni_source             0
electrolyte           0
dtype: int64

[test material_id 중복 개수]
0

[test 결측치 총개수]
0

[test 결측치 상위 컬럼]
material_id           0
material_structure    0
synthesis_method      0
sintering_T1(C)       0
sintering_t1(h)       0
Li_source             0
Co_source             0
Mn_source             0
Ni_source             0
electrolyt